<a href="https://colab.research.google.com/github/sylvite/UCSD-Agentic-AI/blob/main/midterm_timcrnkovic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Midterm Project Starter

**Agentic AI: Building Autonomous Intelligent Systems** &middot; *Individual &middot; due the posted Week 6 deadline*

This is the skeleton every menu option fits into. Read the full spec first &mdash; **`midterm/README.md`** in the course repo &mdash; then make a copy of this notebook and build your tool inside it. Replace this cell with your tool's name and a one-paragraph description of what it does and who it is for.

## The architecture checklist

Your finished notebook must demonstrate all seven. Keep this table; in your repo README you will map each row to the cell that implements it.

| # | Requirement | Taught in |
|---|---|---|
| 1 | Employ good prompt techniques( e.g. Role &middot; Context &middot; Task &middot; Constraints &middot; **structured output** | Week 1 |
| 2 | A **router**, **parallel fan-out**, **OR** **reflection loop** | Weeks 2&ndash;3 |
| 3 | A **crew**: 2+ specialists, named tasks, a **`context=[...]`** handoff | Week 3 |
| 4 | A **custom `@tool`** AND a **prebuilt tool** (e.g. `ScrapeWebsiteTool`) | Week 4 |
| 5 | **`planning=True`** **OR** a **Flow** (`@start` / `@router` / `@listen`) | Week 4 |
| 6 | **Memory** across runs: `memory=True` + embedder, or an episodic log | Weeks 3/5 |
| 7 | A saved **artifact** file produced on every run | &mdash; |

## Grading (20 pts)

| Component | Pts |
|---|---|
| The seven required elements, present and **correctly used** | 8 |
| The tool works end to end and produces its artifact | 4 |
| Design judgment &mdash; the right pattern for each job, least privilege, no decoration | 3 |
| README &mdash; accurate architecture map + AI-use notes | 3 |
| Clean top-to-bottom run with visible outputs | 2 |

Full spec, the domain menu, and submission details are in `midterm/README.md`.

## Setup

Add your key as a Colab secret: get one from [Google AI Studio](https://aistudio.google.com/app/apikey), click the **key icon** in Colab, add a secret named **`GEMINI_API_KEY`**, and enable **Notebook access**. The install pulls CrewAI and its prebuilt-tools package.

In [1]:
!pip install -q crewai crewai-tools

> **Heads-up on the pip output:** an `ERROR: pip's dependency resolver ...` line about `bigframes`/`rich` is **expected in Colab and safe to ignore**. If Colab offers a **"Restart session"** prompt, click it and re-run from setup.

In [2]:
import os

from google.colab import userdata
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from google import genai

load_dotenv()
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")
os.environ["GEMINI_API_KEY"] = api_key
client = genai.Client(api_key=api_key)

# gemini-2.5-flash (not flash-lite): a crew fires many multi-step requests, and the lite model is
# often overloaded (HTTP 503) under that load. num_retries adds a per-call cushion.

## N.B.:  2.5-flash seems to be completely deprecated - at least when I try it - as I get a 404 error every single time.
## so I am using 3.1-flash-lite instead
#llm = LLM(model="gemini/gemini-2.5-flash", api_key=api_key, temperature=0.3, num_retries=5)
llm = LLM(model="gemini/gemini-3.1-flash-lite", api_key=api_key, temperature=0.3, num_retries=5)

# Keep output clean: quiet CrewAI/LiteLLM ERROR logs, hide a legacy-SDK notice, no run traces.
import logging, warnings
for _n in ("crewai.flow.runtime", "LiteLLM", "litellm", "root"):
    logging.getLogger(_n).setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message=r".*google\.generativeai.*")
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Silence CrewAI's tracing notices (the repeating "Tracing Preference" panels) so only the clean
# execution boxes show. The first-time flag is captured when crewai is imported, so we also clear
# it directly on the listener. (Internal API -- guarded.)
try:
    from crewai.events.listeners.tracing.utils import (
        set_suppress_tracing_messages, mark_first_execution_done,
    )
    set_suppress_tracing_messages(True)
    mark_first_execution_done()
    from crewai.events.listeners.tracing.trace_listener import TraceCollectionListener
    if TraceCollectionListener._instance is not None:
        TraceCollectionListener._instance.first_time_handler.is_first_time = False
except Exception:
    pass

print("Setup ready.")

Setup ready.


## 1 &mdash; Tools

*(Checklist items 4: one custom `@tool` whose docstring tells the model exactly what it does and how to call it, plus one prebuilt tool. Only the agent that needs a tool should get it.)*

In [3]:
# Your tools here.
# - prebuilt:     from crewai_tools import ScrapeWebsiteTool
from crewai_tools import ScrapeWebsiteTool
scrape = ScrapeWebsiteTool()


# - custom @tool: from crewai.tools import tool (imported in setup)
# This custom tool will call the public api for clinicaltrials.gov to search for clinical trials based on parameters
import requests
@tool("search_clinical_trials")
def search_clinical_trials(condition: str, max_results: int = 3) -> str:
    """
    Searches the ClinicalTrials.gov database for active or recruiting
    studies related to a specific rare disease condition or phenotype.
    Args:
        condition: The disease name or phenotype keyword (e.g., 'SCN1A encephalopathy')
        max_results: Maximum number of trials to return
    """
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {
        "query.cond": condition,
        "filter.overallStatus": "RECRUITING",
        "pageSize": max_results
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        studies = data.get("studies", [])
        if not studies:
            return "No recruiting clinical trials found for this specific condition."

        formatted_output = []
        for study in studies:
            protocol = study.get("protocolSection", {})
            nct_id = protocol.get("identificationModule", {}).get("nctId", "N/A")
            title = protocol.get("identificationModule", {}).get("briefTitle", "N/A")
            status = protocol.get("statusModule", {}).get("overallStatus", "N/A")

            # Construct the exact human-readable URL for reference or reporting
            study_url = f"https://clinicaltrials.gov/study/{nct_id}"

            formatted_output.append(
                f"- Trial ID: {nct_id}\n  Title: {title}\n  Status: {status}\n  URL: {study_url}\n"
            )

        return "\n".join(formatted_output)

    except Exception as e:
        return f"Error connecting to ClinicalTrials.gov API: {str(e)}"

## 2 &mdash; Agents and tasks

*(Checklist items 1 and 3: specialist agents with Role &middot; Context &middot; Task &middot; Constraints &middot; Format thinking in their role/goal/backstory; named tasks; at least one `context=[...]` handoff. At least one step should return **structured output** via a pydantic schema.)*

In [4]:
# Your agents and tasks here.


# This agent is given the url for a wikipedia page pertaining to a human disease and scrapes the page in order to find any genes that are commonly mutated with that disease
gene_finder = Agent(role="Gene Mutation Researcher", goal="Scrape the names of a disease and mutated genes that are commonly associated with this disease from an assigned web page.",
                   backstory="You discern relevant disease names and associated mutated gene names from a web page.",
                   tools=[scrape], llm=llm)

gene_task = Task(
    description="Read the following specified Wikipedia page with the scrape tool and find the genes commonly mutated in people with the disease described on the page. If there are multiple sich genes, return the gene is the most commonly mutated one in general: {wiki_page}.",
    expected_output="The name of the disease described on the specified Wikipedia page and the name of the gene most commonly mutated in that disease.",
    agent=gene_finder,
)


# This agent calls the clinicaltrials.gov api and searchs for clinical trials given the name of a disease and commonly mutated genes for that disease
trial_matcher = Agent(role="Clinical Trial Matcher", goal="Query clinicaltrials.gov and return a list of clinical trials for a disease/gene combination.",
                   backstory="You call a public api for the U.S. governmental clinical trial reposistory and search for trials recruiting for a specified disease and gene",
                   tools=[search_clinical_trials], llm=llm)

matcher_task = Task(
    description="Using the provided tool, query the api for clinicaltrials.gov and using the name of a disease and the name of a commonly mutated gene for that disease, provided in the context, return a relevant list of clinical trials for that disease/gene which are curreently recruting.",
    expected_output="A list of relevant clinical trials.",
    agent=trial_matcher,
    context=[gene_task],
)

# This agents writes a clinician report of a disease and genese and clinical trials that are recriuiting for that disease/gene combination
medical_writer = Agent(role="Medical Writer", goal="Compiles the list of relevent clinical trials for the disease/gene in question into a final clinician_report.md artifact.",
               backstory="You write prose intended for an audience of medical clinicians.", llm=llm)

writing_task = Task(
    description="Using the provided tool, query the api for clinicaltrials.gov and using the name of a disease and the name of a commonly mutated gene for that disease, provided in the context, return a relevant list of clinical trials for that disease/gene which are curreently recruting.",
    expected_output="An artifact in the form of a markdown file named 'clinician_report.md' containing a list of clinical trials for the specified disease/gene combination.",
    agent=medical_writer,
    context=[gene_task, matcher_task],
)


## 3 &mdash; The crew and its control layer

*(Checklist items 2 and 5: assemble the crew, and put real control flow around or inside it &mdash; a router, fan-out, or reflection loop somewhere in the pipeline, and either `planning=True` on the crew or a Flow governing it. Choose deliberately: if the steps are known, constrain them; if they must be discovered, delegate.)*

In [5]:
from google.genai import errors
import time

MODEL = "gemini-3.1-flash-lite"

# Helper functions to call Gemini to be used with classifier based-routing
def _call(**kwargs):
    """Call Gemini, retrying on transient 429 (rate limit) and 503 (overloaded) errors."""
    for attempt in range(4):  # the first call + up to 3 retries
        try:
            return client.models.generate_content(model=MODEL, **kwargs)
        except errors.ClientError as e:
            # 429 = too many requests (rate limit).
            if e.code == 429 and attempt < 3:
                print("  Rate limited (429). Waiting 60s, then retrying...")
                time.sleep(60)
            else:
                raise
        except errors.ServerError as e:
            # 503 = model temporarily overloaded ("high demand"). A short wait usually clears it.
            if e.code == 503 and attempt < 3:
                print("  Model overloaded (503). Waiting 10s, then retrying...")
                time.sleep(10)
            else:
                raise

# generate: use this when you want plain TEXT back (a sentence, a paragraph).
def generate(prompt: str) -> str:
    """Send a single prompt to Gemini and return the response text."""
    return (_call(contents=prompt).text or "").strip()



# Your crew (and Flow, if you use one) here.
# Colab note: kick off crews with `await crew.kickoff_async()`; Flows run with .kickoff().
import io, contextlib, re, concurrent.futures
from crewai.utilities.planning_handler import CrewPlanner

# Ask CrewAI's planner for the plan -- the same call Crew(planning=True) makes internally.
# The planner runs SYNCHRONOUSLY and Colab has a running event loop, so we run it in a worker
# thread. We also turn the console formatter's verbose OFF just around this call, so the planner's
# raw-metadata "Task Started" box does not show -- the crew runs below keep their clean boxes.
def _make_plan():
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return CrewPlanner(tasks=[gene_task, matcher_task, writing_task], planning_agent_llm=llm)._handle_crew_planning()

try:
    from crewai.events.event_listener import event_listener as _el
    _fmt = _el.formatter
    _prev = _fmt.verbose
    _fmt.verbose = False
except Exception:
    _fmt = None

try:
    with concurrent.futures.ThreadPoolExecutor() as _ex:
        plan = _ex.submit(_make_plan).result()
finally:
    if _fmt is not None:
        _fmt.verbose = _prev

for step_plan in plan.list_of_plans_per_task:
    print(f"=== Plan for Task {step_plan.task_number} ===")
    # put each numbered step on its own line for readability
    print(re.sub(r'(?<=[.")\]])\s?(?=(?:Step\s+\d+[.:]|$d+\.)\s)', "\n", step_plan.plan.strip()).strip())
    print()

=== Plan for Task 1 ===
1. Use the ScrapeWebsiteTool to access and read the content of the provided Wikipedia URL. 2. Analyze the text to identify the primary disease name discussed in the article. 3. Search the text for sections related to genetics, mutations, or molecular basis of the disease. 4. Extract all mentioned genes associated with the disease. 5. Compare the frequency or clinical significance of the identified genes as described in the text to determine the most commonly mutated gene. 6. Synthesize the findings into a clear statement identifying the disease and the primary mutated gene.

=== Plan for Task 2 ===
1. Retrieve the disease name and gene name from the output of the previous task. 2. Construct a search query combining the disease name and the gene name to ensure specificity. 3. Utilize the search_clinical_trials tool with the constructed query. 4. Filter the results returned by the tool to ensure only trials marked as 'recruiting' are included. 5. Compile the list 

## 4 &mdash; Memory

*(Checklist item 6: the tool remembers something between runs &mdash; crew `memory=True` with the Gemini embedder, or an episodic log you write and recall yourself. Say in a comment what gets remembered and why it is useful on the next run.)*

In [6]:
# Your memory layer here.
# Embedder config that works with this stack:
#   embedder={"provider": "google-generativeai",
#             "config": {"api_key": api_key, "model_name": "gemini-embedding-001"}}

#Between runs, store the list of clinical trials that were retrieved for a disease/gene combination
planning_memory_crew = Crew(
    agents=[gene_finder, trial_matcher, medical_writer],
    tasks=[gene_task, matcher_task, writing_task],
    process=Process.sequential,
    planning=True,           # <-- draft a plan before executing
    planning_llm=llm,
    memory=True,  # <-- turn on short-term / long-term / entity memory across runs
    embedder={
        # CrewAI renamed the Gemini embeddings provider: use "google-generativeai" (the Gemini API
        # path) -- "google-vertex" is the separate Vertex AI path, and the old bare "google" is gone.
        "provider": "google-generativeai",
        "config": {"api_key": api_key, "model_name": "gemini-embedding-001"},
    },
    verbose=False,
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 5 &mdash; Run the tool

*(Checklist item 7: one cell that runs the whole pipeline on a real input and saves the artifact file. Run it at least twice before submitting so the memory behavior is visible in the outputs.)*

In [7]:
import io, contextlib, re, concurrent.futures

# Kick off the pipeline, print the result, and write your artifact file.


# We will use an LLM Classifier-based router to determine if a url is valid and it points to a Wikipedia page about a human disease.
# The router will return a simple text response of "VALID" or "INVALID". NO schema or json is needed for that.

# There is one valid url among this list. The rest should be marked as INVALID by the LLM classifier router.
# The valid one will be processed and the rest discarded.
TEST_URLS  = [
    "raindrops and roses",
    "https://www.weather.gov",
    "https://www.baidu.com",
    "https://en.wikipedia.org/wiki/Multiple_myeloma",
    "https://www.amazon.con",
    "https://en.wikipedia.org/wiki/Pickleball"
]

# Define a dummy ROUTES if it's not defined elsewhere, to prevent NameError
try:
    ROUTES
except NameError:
    ROUTES = {
        "valid_disease_wiki": "A valid Wikipedia URL about a human disease.",
        "invalid": "Not a valid URL, or not about a human disease on Wikipedia."
    }

# Url_classifier asks the LLM to determine if a url is valid and points to a wikipedia page about a human disease.
# It returns a string of "VALID" or "INVALID" only.
def url_classifier(test_url: str) -> str:
    """Classify the test url as valid or invalid via the LLM."""
    # The part inside join() is a generator expression: it produces one "- name: description"
    # line per (name, description) pair in ROUTES -- the menu we show the model.
    routes_desc = "\n".join(f"- {name}: {desc}" for name, desc in ROUTES.items())
    result = generate(
        "Classify the test url as either 'VALID' if it is a valid url and it also points to a Wikipedia page about a human disease, or 'INVALID' otherwise.\n\n"
        f"TEST URL: {test_url}",
    )
    return result

# Iterate through test urls, using the llm classifer as a router to determine if the url is valid and points to a wikipedia page about a human disease.
for url in TEST_URLS:
  if (url_classifier(url) == "VALID"):
    print(f"The url {url} is valid\n")
    # Run first time
    plan_result1 = await planning_memory_crew.kickoff_async(inputs={"wiki_page": url})
    print("\nFinal result - First Run:\n", plan_result1)
    # Save the first inal result as a Markdown file artifact
    artifact_filename = "clinician_report1.md"
    with open(artifact_filename, "w", encoding="utf-8") as f:
        f.write(str(plan_result1))
        print(f"\nArtifact successfully saved to {artifact_filename}")

    # Run second time - uses memory stored from first run
    plan_result2 = await planning_memory_crew.kickoff_async(inputs={"wiki_page": url})
    print("\nFinal result - Second Run:\n", plan_result2)

    # Save the second final result as a Markdown file artifact
    artifact_filename = "clinician_report2.md"
    with open(artifact_filename, "w", encoding="utf-8") as f:
        f.write(str(plan_result2))
        print(f"\nArtifact successfully saved to {artifact_filename}")
  else:
    print(f"The url {url} is invalid\n")



The url raindrops and roses is invalid

The url https://www.weather.gov is invalid

The url https://www.baidu.com is invalid

The url https://en.wikipedia.org/wiki/Multiple_myeloma is valid


Final result - First Run:
 # clinician_report.md

# Clinical Trial Report: Multiple Myeloma

**Date:** October 26, 2023
**Target Disease:** Multiple Myeloma
**Focus Genes:** *KRAS*, *NRAS* (Primary); *BRAF* (Secondary/Heterogeneity)

## Overview
Multiple myeloma is characterized by significant genetic heterogeneity. While *KRAS* and *NRAS* mutations are the most frequently identified oncogenic drivers within the RAS pathway in this patient population, *BRAF* mutations also represent a clinically significant subset. The following clinical trials are currently recruiting and investigate targeted therapeutic approaches for patients harboring these specific genetic alterations.

---

## Identified Clinical Trials

### 1. The TG01 Study With TG01/QS-21 Vaccine
*   **Trial ID:** [NCT05841550](https://cl

## Before you submit

- Runs top to bottom in a fresh Colab session with only the `GEMINI_API_KEY` secret.
- The artifact file is produced, and the second run shows the memory doing something.
- No API key pasted in any cell.
- Your repo `README.md` has the description, your ASCII architecture diagram, the checklist table mapped to cells, and your AI-use notes.
- Submit the repo or Colab link via the Canvas assignment.